In [45]:
from langchain_community.document_loaders import WebBaseLoader
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

In [46]:
web_page = "https://en.wikipedia.org/wiki/World_War_II"
loader = WebBaseLoader(web_page)
docs = loader.load()
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/World_War_II', 'title': 'World War II - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nWorld War II - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\t\tPrint/export\n\t\n\n\nDownload as PDFPrintable version\n\n\n\n\n\n\t\tIn other projects\n\t\n\n\nAbstract WikipediaWikimedia CommonsWikinewsWikiquoteWikiversityWikivoyageWikidata item\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate accou

In [47]:
# split the document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)

In [48]:
documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/World_War_II', 'title': 'World War II - Wikipedia', 'language': 'en'}, page_content='World War II - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\t\tPrint/export\n\t\n\n\nDownload as PDFPrintable version\n\n\n\n\n\n\t\tIn other projects\n\t\n\n\nAbstract WikipediaWikimedia CommonsWikinewsWikiquoteWikiversityWikivoyageWikidata item\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\n

In [49]:
# embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()


In [52]:
# Faiss vector db
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(documents=documents, embedding=embeddings)
vectorstoredb

In [53]:
query = "What was the duration of World War II?"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'Start and end dates\nSee also: List of timelines of World War II\nTimelines of World War II\nChronological\nPrelude Events (in Asiain Europe)\nAftermath\n\n\n\n1939\n1940\n1941\n1942\n1943\n1944\n1945\nAftermath\n\n\nBy topic\n\nCauses (Diplomacy)\nDeclarations of war\nBattlesOperations\n\n\nBy theatre\n\nBattle of Europe air operations\nEastern FrontManhattan Project\nUnited Kingdom home front\nSurrender of the Axis armies'

In [54]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    # provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [55]:
# Retrieval chain, document chain
# needs langchain classic package (legacy)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# import chatprompttemplate for creating prompts
from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_template(
#     """
# 		Answer the following questions based only on the provided context:
# 		<context>
# 			{context}
#         </context>
# 	"""
# )

prompt = ChatPromptTemplate.from_messages(
    [  
		("system", "Answer the user's question based only on the provided context:\n\n<context>\n{context}\n</context>"),
		("human", "{input}")
	]
)
document_chain = create_stuff_documents_chain(chat_model, prompt)

In [56]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Answer the user's question based only on the provided context:\n\n<context>\n{context}\n</context>"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='meta-llama/Llama-3.3-70B-Instruct', repetition_penalty=1.03, stop_sequences=[], server_kwargs={}, model_kwargs={}, model='meta-llama/Llama-3.3-70B-Instruct', client=<InferenceClient(model='meta-llama/Llama-3.3-70B-Instruct', timeout=120)>, async_clie

In [57]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": "What was the duration of World War II?",
    "context": [Document(page_content="World War II,[b] or the Second World War (1 September 1939 – 2 September 1945), was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated. Tanks and aircraft played major roles, the latter enabling the strategic bombing of cities and delivery of the only nuclear weapons used in war. World War II was the deadliest conflict in history, causing the death of 60 to 75 million people. Millions died as a result of massacres, starvation, disease, and genocides, including the Holocaust. After the Allied victory, Germany, Austria, Japan, and Korea were occupied, and German and Japanese leaders were tried for war crimes.")]
    
})

'The duration of World War II was from 1 September 1939 to 2 September 1945, approximately 6 years.'

In [59]:
# Input -> retriever -> brings data from vectorstoredb - we are not doing any similarity search
# convert the vectorstoredb into a retriever.
# Retriver is nothing but an interface.
retriever = vectorstoredb.as_retriever()

In [60]:
from langchain_classic.chains import create_retrieval_chain

# document_chain would be responsible for giving the context during retrieval
retriever_chain = create_retrieval_chain(retriever, document_chain)

In [61]:
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E0DDEFF610>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Answer the user's question based only on the provided context:\n\n<context>\n{context}\n</context>"), additional_kwargs={}), HumanMess

In [63]:
# get the response from the LLM
response = retriever_chain.invoke({
    "input": "What was the duration of World War II?"
})
response['answer']

'The provided context does not explicitly state the exact duration of World War II in years, but it mentions the start and end dates as 1939 and 1945, respectively. Based on this information, World War II lasted approximately 6 years.'

In [64]:
response

{'input': 'What was the duration of World War II?',
 'context': [Document(id='2f093c64-c255-4702-a827-331e3b680832', metadata={'source': 'https://en.wikipedia.org/wiki/World_War_II', 'title': 'World War II - Wikipedia', 'language': 'en'}, page_content='Start and end dates\nSee also: List of timelines of World War II\nTimelines of World War II\nChronological\nPrelude Events (in Asiain Europe)\nAftermath\n\n\n\n1939\n1940\n1941\n1942\n1943\n1944\n1945\nAftermath\n\n\nBy topic\n\nCauses (Diplomacy)\nDeclarations of war\nBattlesOperations\n\n\nBy theatre\n\nBattle of Europe air operations\nEastern FrontManhattan Project\nUnited Kingdom home front\nSurrender of the Axis armies'),
  Document(id='796151df-04cc-4e85-b3e2-f2a2f538e136', metadata={'source': 'https://en.wikipedia.org/wiki/World_War_II', 'title': 'World War II - Wikipedia', 'language': 'en'}, page_content='World War II - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n

In [65]:
# get the response from the LLM
response = retriever_chain.invoke({
    "input": "Who started World War II?"
})
response['answer']

'Most historians agree that World War II began with the German invasion of Poland on 1 September 1939, led by Nazi Germany under Adolf Hitler.'

In [66]:
response['context']

[Document(id='08aa012e-d75e-4066-aecc-b68c4144ee79', metadata={'source': 'https://en.wikipedia.org/wiki/World_War_II', 'title': 'World War II - Wikipedia', 'language': 'en'}, page_content="Background\nMain article: Causes of World War II\nAftermath of World War I\nThe League of Nations assembly, held in Geneva, Switzerland (1930)\nWorld War I radically altered the European political map. The most prominent nations of the Central Powers each lost territory in their respective peace treaties at the conclusion of the conflict. New nation-states were created out of the dissolution of the Austro-Hungarian, Ottoman, and Russian Empires.[14]\nTo prevent a future world war, the League of Nations was established in 1920 by the Paris Peace Conference. The organisation's primary function was to prevent armed conflict through collective security, military, and naval disarmament, as well as settling international disputes through peaceful negotiations and arbitration.[15]"),
 Document(id='c413b2f1-